## Manual cluster correction and contextual relabeling

This section starts from the completed `data/entities.csv` export. It does not
rerun embedding, HDBSCAN clustering, article translation, or the earlier cells.

The workflow:

1. creates an immutable `entities_before_manualcheck.csv` snapshot;
2. applies seven reviewed cluster splits using explicit article IDs;
3. validates every reassignment and writes an audit trail;
4. recalculates cluster keywords for the fourteen affected groups;
5. generates contextual English label candidates with `gpt-5.6-sol`;
6. pauses for external review and approval;
7. translates only approved English labels into Norwegian;
8. validates and exports the final dataset.

Run this section from top to bottom. The API checkpoints and review files are
intentionally not overwritten when they already exist.


### 1. Locate the repository and preserve the pre-review dataset

The backup is created only once. Every later run reloads that backup, preventing
manual changes from accumulating if the notebook cells are executed more than once.
SHA-256 hashes make the provenance visible without modifying either CSV.


In [23]:
from pathlib import Path
from collections import Counter
from hashlib import sha256
from shutil import copy2
import json
import os
import time

import numpy as np
import pandas as pd


def locate_repo_root():
    '''Locate the repository whether the notebook runs from root or data/.'''
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]

    for candidate in candidates:
        if (
            (candidate / "data").is_dir()
            and (candidate / "public").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate a repository containing data/ and public/."
    )


def file_sha256(path):
    digest = sha256()
    with path.open("rb") as file_handle:
        for block in iter(lambda: file_handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


REPO_ROOT = locate_repo_root()
DATA_DIR = REPO_ROOT / "data"
PUBLIC_DIR = REPO_ROOT / "public"
SOURCE_ENTITIES_PATH = DATA_DIR / "entities.csv"
BACKUP_ENTITIES_PATH = DATA_DIR / "entities_before_manualcheck.csv"

if not SOURCE_ENTITIES_PATH.is_file():
    raise FileNotFoundError(SOURCE_ENTITIES_PATH)

backup_created = False

if not BACKUP_ENTITIES_PATH.exists():
    copy2(SOURCE_ENTITIES_PATH, BACKUP_ENTITIES_PATH)
    backup_created = True

source_hash = file_sha256(SOURCE_ENTITIES_PATH)
backup_hash = file_sha256(BACKUP_ENTITIES_PATH)

if backup_created:
    assert source_hash == backup_hash

entities_original = pd.read_csv(
    BACKUP_ENTITIES_PATH,
    dtype={"cluster": str}
)
entities_original["id"] = entities_original["id"].astype(str)

EXPECTED_ROWS = 10946
EXPECTED_ORIGINAL_CLUSTERS = 167

assert len(entities_original) == EXPECTED_ROWS
assert entities_original["cluster"].nunique() == EXPECTED_ORIGINAL_CLUSTERS
assert not entities_original["id"].duplicated().any()

print("Repository:", REPO_ROOT)
print("Backup created:", backup_created)
print("Current entities hash:", source_hash)
print("Protected backup hash:", backup_hash)
print("Rows:", len(entities_original))
print("Original clusters:", entities_original["cluster"].nunique())


Repository: /Users/tina/Documents/GitHub/svalbardposten_weathermap_final
Backup created: False
Current entities hash: 3ae9a38d6d6239047d1ee4153e8fb531c41224d9b706de20940f065909fc7b33
Protected backup hash: ddd131a722e8472de716b43e321f9f42b254e468a440931794c78e1445c20ec6
Rows: 10946
Original clusters: 167


### 2. Apply the seven reviewed splits

The final assignments use article IDs rather than recalculating K-means or applying
coordinate thresholds. This makes the result deterministic and independently auditable.

For each split, the original ID stays with one topic and the listed articles move to
a new unused ID. Cluster `p_72` is deliberately divided into 21 water articles and
11 remaining articles. The 11 remain together for contextual labeling; they are not
divided into a four-article career cluster or reassigned elsewhere.


In [24]:
SPLIT_MOVES = {
    "p_115": {
        "source_cluster": "p_1",
        "provisional_topic": "Svalbard earthquakes",
        "article_ids": [
            "537362", "508555", "492193", "213410", "170889",
            "164156", "146910", "146899", "146785"
        ]
    },
    "p_116": {
        "source_cluster": "p_2",
        "provisional_topic": "Mark Sabbatini and Icepeople",
        "article_ids": [
            "509439", "485036", "220961", "215640", "215623",
            "215436", "207536", "203592", "189204", "183838",
            "451020", "440142", "181732", "178681", "166413",
            "165302", "162826", "155970", "146651"
        ]
    },
    "p_117": {
        "source_cluster": "p_49",
        "provisional_topic": "Field inspections and environmental enforcement",
        "article_ids": [
            "537337", "505151", "491351", "486149", "216271",
            "209111", "198258", "169391", "159319", "148549"
        ]
    },
    "p_118": {
        "source_cluster": "p_72",
        "provisional_topic": "Contextual labeling required",
        "article_ids": [
            "510114", "505079", "491759", "489138", "486964",
            "484053", "230941", "220482", "214708", "212600",
            "201581"
        ]
    },
    "p_119": {
        "source_cluster": "p_77",
        "provisional_topic": "Trails and visitor infrastructure",
        "article_ids": [
            "191232", "189582", "182325", "165504", "164224",
            "162215", "160427"
        ]
    },
    "n_52": {
        "source_cluster": "n_9",
        "provisional_topic": "Hunting and fishing regulations",
        "article_ids": [
            "140033", "139355", "134012", "131775", "125135",
            "124171", "121480", "118749", "116609", "114245",
            "112213", "111636", "111199", "110841", "108709",
            "107500", "107074", "105079", "104769", "101410",
            "101278"
        ]
    },
    "p_120": {
        "source_cluster": "p_104",
        "provisional_topic": "Tourism leadership and sustainability",
        "article_ids": [
            "494269", "222928", "222488", "215743", "211792",
            "189818", "187444", "177933", "175360", "174720",
            "172796", "172523", "171296", "170432", "151543",
            "148016", "146929"
        ]
    }
}

EXPECTED_SPLIT_COUNTS = {
    "p_1": 26,
    "p_115": 9,
    "p_2": 25,
    "p_116": 19,
    "p_49": 23,
    "p_117": 10,
    "p_72": 21,
    "p_118": 11,
    "p_77": 30,
    "p_119": 7,
    "n_9": 37,
    "n_52": 21,
    "p_104": 13,
    "p_120": 17
}

new_cluster_ids = set(SPLIT_MOVES)
existing_cluster_ids = set(entities_original["cluster"])
collisions = new_cluster_ids & existing_cluster_ids

if collisions:
    raise ValueError(f"New cluster IDs already exist: {sorted(collisions)}")

moved_id_lists = [
    article_id
    for split in SPLIT_MOVES.values()
    for article_id in split["article_ids"]
]

if len(moved_id_lists) != len(set(moved_id_lists)):
    raise ValueError("An article ID appears in more than one split.")

entities_manual = entities_original.copy()
original_cluster_by_id = entities_manual.set_index("id")["cluster"].to_dict()
audit_rows = []

for new_cluster, split in SPLIT_MOVES.items():
    source_cluster = split["source_cluster"]
    article_ids = set(split["article_ids"])
    located = entities_manual["id"].isin(article_ids)

    if located.sum() != len(article_ids):
        missing = sorted(article_ids - set(entities_manual.loc[located, "id"]))
        raise ValueError(
            f"Missing IDs for {source_cluster} -> {new_cluster}: {missing}"
        )

    wrong_source = entities_manual.loc[
        located & (entities_manual["cluster"] != source_cluster),
        ["id", "cluster"]
    ]

    if not wrong_source.empty:
        raise ValueError(
            f"IDs assigned to an unexpected source cluster:\n{wrong_source}"
        )

    for article_id in sorted(article_ids):
        audit_rows.append({
            "id": article_id,
            "original_cluster": source_cluster,
            "revised_cluster": new_cluster,
            "assignment_reason": split["provisional_topic"],
            "review_status": "reviewed"
        })

    entities_manual.loc[located, "cluster"] = new_cluster

    # Do not let moved articles retain the parent cluster's old label.
    entities_manual.loc[
        located,
        ["cluster_subject_en", "cluster_subject_no"]
    ] = pd.NA

split_audit = pd.DataFrame(audit_rows).sort_values(
    ["original_cluster", "revised_cluster", "id"]
)

actual_split_counts = entities_manual[
    entities_manual["cluster"].isin(EXPECTED_SPLIT_COUNTS)
]["cluster"].value_counts().to_dict()

assert actual_split_counts == EXPECTED_SPLIT_COUNTS
assert len(entities_manual) == EXPECTED_ROWS
assert not entities_manual["id"].duplicated().any()
assert entities_manual["cluster"].nunique() == 174

SPLIT_AUDIT_PATH = DATA_DIR / "manual_cluster_split_audit.csv"
SPLIT_CHECKPOINT_PATH = DATA_DIR / "entities_manual_split_checkpoint.csv"

split_audit.to_csv(SPLIT_AUDIT_PATH, index=False, encoding="utf-8")
entities_manual.to_csv(
    SPLIT_CHECKPOINT_PATH,
    index=False,
    encoding="utf-8"
)

print("Rows preserved:", len(entities_manual))
print("Clusters after splits:", entities_manual["cluster"].nunique())
print("Moved articles:", len(split_audit))
print("Audit:", SPLIT_AUDIT_PATH)
print("Split checkpoint:", SPLIT_CHECKPOINT_PATH)

pd.Series(actual_split_counts).sort_index()


Rows preserved: 10946
Clusters after splits: 174
Moved articles: 94
Audit: /Users/tina/Documents/GitHub/svalbardposten_weathermap_final/data/manual_cluster_split_audit.csv
Split checkpoint: /Users/tina/Documents/GitHub/svalbardposten_weathermap_final/data/entities_manual_split_checkpoint.csv


n_52     21
n_9      37
p_1      26
p_104    13
p_115     9
p_116    19
p_117    10
p_118    11
p_119     7
p_120    17
p_2      25
p_49     23
p_72     21
p_77     30
dtype: int64

### 3. Inspect the split membership before recalculating keywords

This is a validation gate, not a labeling step. Review the counts, coordinate ranges,
and titles. The provisional topics above are documentation only and will not become
final labels.


In [25]:
split_membership_review = (
    entities_manual[
        entities_manual["cluster"].isin(EXPECTED_SPLIT_COUNTS)
    ]
    .groupby("cluster")
    .agg(
        article_count=("id", "size"),
        x_min=("x", "min"),
        x_max=("x", "max"),
        y_min=("y", "min"),
        y_max=("y", "max"),
        sample_titles=("title_en", lambda values: list(values.head(8)))
    )
    .reset_index()
    .sort_values("cluster")
)

split_membership_review


,cluster,article_count,x_min,x_max,y_min,y_max,sample_titles
0,n_52,21,783,789,474,492,"[These Are Going Reindeer Hunting, On Locally ..."
1,n_9,37,752,774,485,505,"[Cabin Door Left Open, Continues at Kapp Wijk,..."
2,p_1,26,265,271,423,426,"[A new version of the mystery of the Iron Bed,..."
3,p_104,13,535,542,315,327,[Multiple violations of the Working Environmen...
4,p_115,9,258,262,464,466,"[Earthquake near Svalbard, 4.7-magnitude earth..."
5,p_116,19,312,329,400,410,[You have to seize the opportunities that come...
6,p_117,10,680,687,552,560,"[The Field Inspectors Are in Place, “A dog was..."
7,p_118,11,498,510,376,396,[“We have uncovered breaches of the sanctions ...
8,p_119,7,591,600,301,315,"[Delays Opening of Local Office, Plans Viewing..."
9,p_120,17,533,549,321,324,"[Back to normal, or an all-time high?, Gets Lo..."


### 4. Recalculate keywords for the affected clusters

The completed `entities.csv` does not contain the full article lemmas used by the
earlier clustering notebook. It does contain aligned Norwegian and English
article-level TF-IDF keywords. For the fourteen affected groups, this section ranks
those aligned keyword pairs by weighted frequency. Earlier positions in each
article's keyword list receive more weight.

Untouched clusters retain their existing cluster keywords. This avoids silently
changing the keyword methodology for all other clusters.


In [26]:
def parse_keyword_list(value):
    if isinstance(value, list):
        return [str(item).strip() for item in value if str(item).strip()]

    if pd.isna(value):
        return []

    parsed = json.loads(str(value))

    if not isinstance(parsed, list):
        raise ValueError(f"Expected a JSON list, received: {type(parsed)}")

    return [str(item).strip() for item in parsed if str(item).strip()]


def aggregate_aligned_keywords(group, top_n=15):
    pair_scores = Counter()
    pair_counts = Counter()
    first_seen = {}
    preferred_display = {}
    sequence = 0

    for _, row in group.iterrows():
        keywords_no = parse_keyword_list(
            row["article_keywords_no"]
        )
        keywords_en = parse_keyword_list(
            row["article_keywords_en"]
        )

        if len(keywords_no) != len(keywords_en):
            raise ValueError(
                f"Keyword length mismatch for article {row['id']}: "
                f"{len(keywords_no)} Norwegian vs "
                f"{len(keywords_en)} English"
            )

        for rank, (keyword_no, keyword_en) in enumerate(
            zip(keywords_no, keywords_en)
        ):
            keyword_no = keyword_no.strip()
            keyword_en = keyword_en.strip()

            if not keyword_no or not keyword_en:
                continue

            normalized_pair = (
                keyword_no.casefold(),
                keyword_en.casefold()
            )

            pair_scores[normalized_pair] += 1.0 / (rank + 1)
            pair_counts[normalized_pair] += 1
            first_seen.setdefault(normalized_pair, sequence)

            # Retain the first encountered spelling for display.
            preferred_display.setdefault(
                normalized_pair,
                (keyword_no, keyword_en)
            )

            sequence += 1

    ranked_pairs = sorted(
        pair_scores,
        key=lambda pair: (
            -pair_scores[pair],
            -pair_counts[pair],
            first_seen[pair]
        )
    )

    selected_pairs = []
    used_no = set()
    used_en = set()

    for normalized_pair in ranked_pairs:
        keyword_no, keyword_en = preferred_display[
            normalized_pair
        ]

        normalized_no = keyword_no.casefold()
        normalized_en = keyword_en.casefold()

        # Prevent either language from repeating the same concept.
        if normalized_no in used_no:
            continue

        if normalized_en in used_en:
            continue

        selected_pairs.append(
            (keyword_no, keyword_en)
        )
        used_no.add(normalized_no)
        used_en.add(normalized_en)

        if len(selected_pairs) == top_n:
            break

    if len(selected_pairs) < top_n:
        raise ValueError(
            f"Only {len(selected_pairs)} distinct aligned "
            "keyword pairs are available."
        )

    return {
        "top_keywords_no": [
            pair[0] for pair in selected_pairs
        ],
        "top_keywords_en": [
            pair[1] for pair in selected_pairs
        ]
    }


affected_clusters = set(EXPECTED_SPLIT_COUNTS)
recalculated_keyword_rows = []

for cluster_id in sorted(affected_clusters):
    cluster_rows = entities_manual[
        entities_manual["cluster"] == cluster_id
    ]
    recalculated = aggregate_aligned_keywords(cluster_rows, top_n=15)

    keywords_no_json = json.dumps(
        recalculated["top_keywords_no"],
        ensure_ascii=False
    )
    keywords_en_json = json.dumps(
        recalculated["top_keywords_en"],
        ensure_ascii=False
    )

    cluster_mask = entities_manual["cluster"] == cluster_id
    entities_manual.loc[cluster_mask, "top_keywords_no"] = keywords_no_json
    entities_manual.loc[cluster_mask, "top_keywords_en"] = keywords_en_json

    recalculated_keyword_rows.append({
        "cluster": cluster_id,
        "article_count": len(cluster_rows),
        **recalculated
    })

recalculated_keywords_review = pd.DataFrame(
    recalculated_keyword_rows
).sort_values("cluster")

for _, row in recalculated_keywords_review.iterrows():
    assert len(row["top_keywords_no"]) == 15
    assert len(row["top_keywords_en"]) == 15

entities_manual.to_csv(
    SPLIT_CHECKPOINT_PATH,
    index=False,
    encoding="utf-8"
)

recalculated_keywords_review


,cluster,article_count,top_keywords_no,top_keywords_en
0,n_52,21,"[dyr, jeger, jakt, ljff, ærfugl, bestand, vann...","[animal, hunter, hunting, ljff, eider, populat..."
1,n_9,37,"[lund, soleim, stasjon, farmhamna, austfjordne...","[Lund, Soleim, station, Farmhamna, Austfjordne..."
2,p_1,26,"[film, holm, kaldnes, småfilme, gruve, bilderk...","[film, Holm, Kaldnes, short film, mine, video ..."
3,p_104,13,"[tilsyn, arbeidstilsynet, arbeidstid, hms, joh...","[inspection, Labour Inspection Authority, work..."
4,p_115,9,"[jordskjelv, skjelv, richter, norsar, styrke, ...","[earthquake, quake, Richter, Norsar, magnitude..."
5,p_116,19,"[sabbatini, mark, icepeople, avis, penge, hår,...","[Sabbatini, mark, Icepeople, newspaper, money,..."
6,p_117,10,"[feltinspektør, kongsfjorden, lutnæs, jobb, ha...","[field inspector, Kongsfjorden, Lutnæs, job, H..."
7,p_118,11,"[kontroll, personkontroll, kriminalitet, tolle...","[inspection, identity checks, crime, customs o..."
8,p_119,7,"[sti, nfd, krystad, city, interimstyr, tillate...","[trail, NFD, Krystad, city, interim board, per..."
9,p_120,17,"[brunvoll, reiselivssjef, strømnes, ronny, vis...","[Brunvoll, tourism director, Strømnes, Ronny, ..."


### 5. Build richer context for every cluster

Labels will be regenerated for all 174 clusters. Each request includes bilingual
cluster keywords and spatially varied article examples with titles, subtitles, and
article-level keywords. The old label is deliberately excluded so it cannot anchor
the model to awkward previous wording.


In [27]:
def representative_articles(group, sample_size=12):
    working = group.copy()
    working["distance_from_centroid"] = np.sqrt(
        (working["x"] - working["x"].mean()) ** 2
        + (working["y"] - working["y"].mean()) ** 2
    )

    selected_indices = []

    def add_index(index):
        if index not in selected_indices:
            selected_indices.append(index)

    if len(working):
        add_index(working["distance_from_centroid"].idxmin())
        add_index(working["distance_from_centroid"].idxmax())
        add_index(working["x"].idxmin())
        add_index(working["x"].idxmax())
        add_index(working["y"].idxmin())
        add_index(working["y"].idxmax())

    ordered_indices = working.sort_values(
        ["x", "y", "id"]
    ).index.to_list()

    if ordered_indices:
        positions = np.linspace(
            0,
            len(ordered_indices) - 1,
            min(sample_size, len(ordered_indices)),
            dtype=int
        )
        for position in positions:
            add_index(ordered_indices[position])

    for index in ordered_indices:
        if len(selected_indices) >= sample_size:
            break
        add_index(index)

    selected = working.loc[selected_indices[:sample_size]]
    records = []

    for _, row in selected.iterrows():
        records.append({
            "id": str(row["id"]),
            "x": float(row["x"]),
            "y": float(row["y"]),
            "title_en": str(row["title_en"]),
            "subtitle_en": str(row["subtitle_en"]),
            "article_keywords_en": parse_keyword_list(
                row["article_keywords_en"]
            )
        })

    return records

def unique_keywords_for_context(value):
    """Remove case-insensitive duplicates while preserving order."""
    keywords = parse_keyword_list(value)
    unique_keywords = []
    seen = set()

    for keyword in keywords:
        normalized = keyword.casefold().strip()

        if normalized and normalized not in seen:
            unique_keywords.append(keyword)
            seen.add(normalized)

    return unique_keywords

label_contexts = []

for cluster_id, group in entities_manual.groupby("cluster", sort=True):
    label_contexts.append({
        "cluster": str(cluster_id),
        "article_count": int(len(group)),
        "top_keywords_no": unique_keywords_for_context(
            group.iloc[0]["top_keywords_no"]
        ),
        "top_keywords_en": unique_keywords_for_context(
            group.iloc[0]["top_keywords_en"]
        ),
        "x_range": [float(group["x"].min()), float(group["x"].max())],
        "y_range": [float(group["y"].min()), float(group["y"].max())],
        "representative_articles": representative_articles(
            group,
            sample_size=min(20, len(group))
        )
    })

assert len(label_contexts) == 174
assert len({item["cluster"] for item in label_contexts}) == 174

label_context_preview = pd.DataFrame([
    {
        "cluster": context["cluster"],
        "article_count": context["article_count"],
        "top_keywords_en": context["top_keywords_en"],
        "sample_titles": [
            article["title_en"]
            for article in context["representative_articles"][:5]
        ]
    }
    for context in label_contexts
])

label_context_preview.head()


,cluster,article_count,top_keywords_en,sample_titles
0,n_0,30,"[sun, March, solar eclipse, hospital stairs, R...","[Daylight Is Returning, One Thousand and One N..."
1,n_1,47,"[hospital, north, UNN, doctor, Tromsø, health,...","[Longyearbyen to Get Its Own Psychologist, Jet..."
2,n_10,41,"[action, auction, money, Sandvik, water, trip,...","[Perspective, Dogs and children in perfect har..."
3,n_11,46,"[gymnastics, Barentsburg, activity, coach, tea...",[This Year’s Tyfus Award Goes to Helle Jakobse...
4,n_12,48,"[May, church, youth, commemoration, Utøya, spe...","[Compared with the North Sea Divers, “Santa Cl..."


In [28]:
next(
    context
    for context in label_contexts
    if context["cluster"] == "p_118"
)

{'cluster': 'p_118',
 'article_count': 11,
 'top_keywords_no': ['kontroll',
  'personkontroll',
  'kriminalitet',
  'toller',
  'vare',
  'tolletat',
  'tolletaten',
  'dyrstad',
  'hotell',
  'larsen',
  'linn',
  'justis',
  'jobb',
  'vareførsel',
  'hagen'],
 'top_keywords_en': ['inspection',
  'identity checks',
  'crime',
  'customs officer',
  'goods',
  'Customs Administration',
  'Norwegian Customs Service',
  'Dyrstad',
  'hotel',
  'Larsen',
  'Linn',
  'justice',
  'job',
  'transport of goods',
  'Hagen'],
 'x_range': [498.0, 510.0],
 'y_range': [376.0, 396.0],
 'representative_articles': [{'id': '486964',
   'x': 502.0,
   'y': 386.0,
   'title_en': 'Increased passenger checks at the airport',
   'subtitle_en': 'The Governor of Svalbard is increasing staff as part of efforts to strengthen identity checks on Svalbard.',
   'article_keywords_en': ['identity checks',
    'checks',
    'Bredli',
    'crime',
    'justice',
    'regulations',
    'goods transport',
    'chief 

### 6. Generate contextual English label candidates

This cell uses `gpt-5.6-sol` with medium reasoning. It saves each successful result
immediately and resumes only when the model, prompt version, and cluster context hash
still match. A model response can mark a cluster incoherent instead of disguising a
mixed cluster with an artificial compound label.


In [29]:
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI
from tqdm.auto import tqdm

dotenv_path = find_dotenv(usecwd=True)

if not dotenv_path:
    raise RuntimeError("No .env file found from the current working directory.")

load_dotenv(dotenv_path, override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise RuntimeError("OPENAI_API_KEY was not found in the environment.")

client = OpenAI(api_key=api_key)

LABEL_MODEL = "gpt-5.6-sol"
LABEL_REASONING_EFFORT = "medium"
LABEL_PROMPT_VERSION = "contextual-cluster-label-v1"
LABEL_CHECKPOINT_PATH = DATA_DIR / "cluster_labels_gpt56sol_checkpoint.json"
LABEL_CANDIDATES_PATH = DATA_DIR / "cluster_labels_en_candidates.csv"
LABEL_REVIEW_PATH = DATA_DIR / "cluster_labels_en_for_review.csv"

LABEL_INSTRUCTIONS = '''
You are a senior topic-taxonomy editor for a bilingual Norwegian newspaper
archive about Svalbard. Infer the single shared subject represented by the
supplied cluster, using the cluster keywords together with the representative
article titles, subtitles, article keywords, and coordinate coverage.

Produce a concise, natural English topic label of two to five words. The label
must describe what the articles are actually about, not merely concatenate
frequent words. Prefer an established subject phrase a newspaper editor or
archivist would use. Do not use a person's name unless that person is genuinely
the central subject of the cluster. Do not add Arctic, Polar, Norwegian, or
Svalbard unless that geographic qualification is necessary to distinguish the
subject. Avoid vague endings such as Activities, Developments, Opportunities,
Matters, or Issues. Do not hide unrelated topics behind an artificial X and Y
label.

Also judge whether the supplied articles form one coherent topic. If they do
not, set coherent to false, lower the confidence, and explain the competing
subjects in the rationale. Still provide the least misleading candidate label
for review. Do not translate the label into Norwegian in this step.
'''.strip()


def stable_json_hash(value):
    serialized = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":")
    )
    return sha256(serialized.encode("utf-8")).hexdigest()


def save_json_atomic(value, path):
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    with temporary_path.open("w", encoding="utf-8") as output_file:
        json.dump(value, output_file, ensure_ascii=False, indent=2)
    temporary_path.replace(path)


if LABEL_CHECKPOINT_PATH.exists():
    with LABEL_CHECKPOINT_PATH.open("r", encoding="utf-8") as input_file:
        label_checkpoint = json.load(input_file)
else:
    label_checkpoint = {}

label_results = {}
failed_label_clusters = []

for context in tqdm(label_contexts, desc="Generating contextual labels"):
    cluster_id = context["cluster"]
    context_hash = stable_json_hash(context)
    existing = label_checkpoint.get(cluster_id)

    if (
        isinstance(existing, dict)
        and existing.get("model") == LABEL_MODEL
        and existing.get("prompt_version") == LABEL_PROMPT_VERSION
        and existing.get("context_hash") == context_hash
        and existing.get("cluster_subject_en")
    ):
        label_results[cluster_id] = existing
        continue

    result = None

    for attempt in range(1, 4):
        try:
            response = client.responses.create(
                model=LABEL_MODEL,
                reasoning={"effort": LABEL_REASONING_EFFORT},
                instructions=LABEL_INSTRUCTIONS,
                input=json.dumps(context, ensure_ascii=False),
                text={
                    "format": {
                        "type": "json_schema",
                        "name": "cluster_label_review",
                        "strict": True,
                        "schema": {
                            "type": "object",
                            "properties": {
                                "cluster_subject_en": {"type": "string"},
                                "coherent": {"type": "boolean"},
                                "confidence": {
                                    "type": "string",
                                    "enum": ["high", "medium", "low"]
                                },
                                "rationale": {"type": "string"}
                            },
                            "required": [
                                "cluster_subject_en",
                                "coherent",
                                "confidence",
                                "rationale"
                            ],
                            "additionalProperties": False
                        }
                    }
                },
                max_output_tokens=500
            )

            parsed = json.loads(response.output_text)
            label = parsed["cluster_subject_en"].strip()

            if not label:
                raise ValueError("The model returned an empty label.")

            result = {
                "cluster": cluster_id,
                "article_count": context["article_count"],
                "cluster_subject_en": label,
                "coherent": bool(parsed["coherent"]),
                "confidence": parsed["confidence"],
                "rationale": parsed["rationale"].strip(),
                "model": LABEL_MODEL,
                "prompt_version": LABEL_PROMPT_VERSION,
                "context_hash": context_hash
            }
            break

        except Exception as error:
            print(
                f"Attempt {attempt} failed for {cluster_id}: {error}"
            )
            if attempt < 3:
                time.sleep(2 ** attempt)

    if result is None:
        failed_label_clusters.append(cluster_id)
        continue

    label_checkpoint[cluster_id] = result
    label_results[cluster_id] = result
    save_json_atomic(label_checkpoint, LABEL_CHECKPOINT_PATH)

if failed_label_clusters:
    raise RuntimeError(
        "English labeling failed for: "
        + ", ".join(sorted(failed_label_clusters))
    )

label_candidates = pd.DataFrame(
    [label_results[context["cluster"]] for context in label_contexts]
).sort_values("cluster")

label_candidates["representative_titles"] = label_candidates["cluster"].map({
    context["cluster"]: json.dumps(
        [
            article["title_en"]
            for article in context["representative_articles"]
        ],
        ensure_ascii=False
    )
    for context in label_contexts
})

label_candidates.to_csv(
    LABEL_CANDIDATES_PATH,
    index=False,
    encoding="utf-8"
)

if not LABEL_REVIEW_PATH.exists():
    label_review = label_candidates.copy()
    label_review["approved_cluster_subject_en"] = ""
    label_review["review_status"] = "pending"
    label_review["review_notes"] = ""
    label_review.to_csv(
        LABEL_REVIEW_PATH,
        index=False,
        encoding="utf-8"
    )
    review_file_created = True
else:
    review_file_created = False

print("English candidates:", len(label_candidates))
print("Incoherent candidates:", int((~label_candidates["coherent"]).sum()))
print("Low-confidence candidates:", int((label_candidates["confidence"] == "low").sum()))
print("Candidate file:", LABEL_CANDIDATES_PATH)
print("Review file:", LABEL_REVIEW_PATH)
print("Review file created:", review_file_created)

label_candidates.sort_values(
    ["coherent", "confidence", "cluster"],
    ascending=[True, True, True]
).head(25)


Generating contextual labels:   0%|          | 0/174 [00:00<?, ?it/s]

English candidates: 174
Incoherent candidates: 40
Low-confidence candidates: 12
Candidate file: /Users/tina/Documents/GitHub/svalbardposten_weathermap_final/data/cluster_labels_en_candidates.csv
Review file: /Users/tina/Documents/GitHub/svalbardposten_weathermap_final/data/cluster_labels_en_for_review.csv
Review file created: False


,cluster,article_count,cluster_subject_en,coherent,confidence,rationale,model,prompt_version,context_hash,representative_titles
15,n_22,49,Longyearbyen businesses,False,low,"The cluster broadly concerns local commerce, b...",gpt-5.6-sol,contextual-cluster-label-v1,034e404d4e3697bded101549726fb7788f0b29cde1d635...,"[""How About a Glass?"", ""Relaxing back in the b..."
34,n_4,52,Space science infrastructure,False,low,The dominant thread is infrastructure for sate...,gpt-5.6-sol,contextual-cluster-label-v1,95e3cea4a1380e90ea0cc90cd7131fcd4ad2c66ea31979...,"[""Radon Measurements Completed"", ""Possible thi..."
76,p_119,7,Sherpa Trail Proposal,False,low,Three articles concern the proposed Sherpa-bui...,gpt-5.6-sol,contextual-cluster-label-v1,06f1b8adf6e6c7108cb13ac036fc72f75f4b62aded3499...,"[""Plans Viewing Platform on Sukkertoppen"", ""Co..."
93,p_26,34,Svalbard plant life,False,low,"The dominant subject is local flora, including...",gpt-5.6-sol,contextual-cluster-label-v1,0d0ded8c4703f79d516a659160f746af59d0551e6fad3a...,"[""“Looking after so many plants is a lot of wo..."
103,p_35,30,Spitsbergen Revue Group,False,low,"The dominant subject is the revue troupe, incl...",gpt-5.6-sol,contextual-cluster-label-v1,fbcd6a11e13196b41047bdbcb76c5cfca29fa9fbceb2ab...,"[""Considered Cancelling This Year’s Revue"", ""P..."
117,p_48,162,Polar exploration heritage,False,low,"Many articles concern exploration history, tra...",gpt-5.6-sol,contextual-cluster-label-v1,308415e44c50580bc864934894c0eb5ec9af24df74f5ad...,"[""In the Shadow of Polar History"", ""People and..."
119,p_5,156,Community Commentary,False,low,The cluster is linked more by commentary and f...,gpt-5.6-sol,contextual-cluster-label-v1,bd5b88e4856341868c7559a38f69d7aea802c50c62b424...,"[""Tourists of the future"", ""The Blue Mountains..."
121,p_51,107,Svalbard environmental research,False,low,"The dominant theme is research on fjords, mari...",gpt-5.6-sol,contextual-cluster-label-v1,1285d0d2d7e8486041e7d9cb002018dae09f4d6c79661e...,"[""The paradox of living on Svalbard"", ""The bat..."
123,p_53,75,Governor’s Office,False,low,The cluster combines two substantial subjects:...,gpt-5.6-sol,contextual-cluster-label-v1,cbf41a1bdf3b0fd22cc7aaa273d0879d5848d0ff3fea38...,"[""“It Will Probably Be a Challenge”"", ""“More W..."
126,p_56,58,Remote station life,False,low,The largest identifiable strand concerns crews...,gpt-5.6-sol,contextual-cluster-label-v1,dc7d88bb653e35df8154ac7815c97ace7be24a527b4efb...,"[""Sandra picks up the finest ingredients at th..."


## STOP: review the English labels before continuing

Share `data/cluster_labels_en_for_review.csv` for online review. For every row:

- enter the accepted or revised wording in `approved_cluster_subject_en`;
- set `review_status` to `approved`;
- optionally record the reason for a change in `review_notes`.

Do not run the translation or final export cells until all 174 labels are approved.
Rerunning the generation cell will update the candidates file but will not overwrite
the existing review file.


In [30]:
label_revisions = {
    "n_3": "Rabies and Animal Health",
    "n_4": "Space Research Infrastructure",
    "n_22": "Retail and Hospitality Businesses",
    "n_46": "Maritime Operations and Incidents",
    "n_48": "Rescue and Police Operations",
    "p_1": "Svalbard History Films",
    "p_2": "Old Hospital Property Disputes",
    "p_5": "Community Commentary and Features",
    "p_35": "Revue and Cultural Awards",
    "p_48": "Exploration History and Heritage",
    "p_56": "Bjørnøya and Hopen Stations",
    "p_72": "Drinking Water Safety",
    "p_82": "Alcohol Licensing and Café Leases",
    "p_97": "Waste and Vehicle Disposal",
    "p_116": "Mark Sabbatini and Icepeople",
    "p_120": "Tourism Management and Sustainability"
}

approved_labels = pd.read_csv(
    LABEL_REVIEW_PATH,
    dtype={"cluster": str}
)

approved_labels[
    "approved_cluster_subject_en"
] = approved_labels["cluster_subject_en"]

approved_labels.loc[
    approved_labels["cluster"].isin(label_revisions),
    "approved_cluster_subject_en"
] = (
    approved_labels.loc[
        approved_labels["cluster"].isin(label_revisions),
        "cluster"
    ].map(label_revisions)
)

approved_labels["review_status"] = "approved"

approved_labels.loc[
    approved_labels["cluster"].isin(label_revisions),
    "review_notes"
] = "English label revised during manual review."

approved_labels.loc[
    ~approved_labels["cluster"].isin(label_revisions),
    "review_notes"
] = "Generated English label approved."

assert len(approved_labels) == 174
assert approved_labels["cluster"].nunique() == 174
assert approved_labels[
    "approved_cluster_subject_en"
].str.strip().ne("").all()
assert approved_labels[
    "review_status"
].eq("approved").all()

approved_labels.to_csv(
    LABEL_REVIEW_PATH,
    index=False,
    encoding="utf-8"
)

print("Approved labels:", len(approved_labels))
print("Revised labels:", len(label_revisions))

Approved labels: 174
Revised labels: 16


### 7. Load and validate the approved English labels

This cell deliberately fails if even one cluster is missing, still pending, or has an
empty approved label.


In [31]:
approved_labels = pd.read_csv(
    LABEL_REVIEW_PATH,
    dtype={"cluster": str}
)

required_review_columns = {
    "cluster",
    "approved_cluster_subject_en",
    "review_status"
}
missing_review_columns = required_review_columns - set(approved_labels.columns)

if missing_review_columns:
    raise ValueError(
        f"Review file is missing columns: {sorted(missing_review_columns)}"
    )

approved_labels["approved_cluster_subject_en"] = (
    approved_labels["approved_cluster_subject_en"]
    .fillna("")
    .astype(str)
    .str.strip()
)
approved_labels["review_status"] = (
    approved_labels["review_status"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

current_cluster_ids = set(entities_manual["cluster"].astype(str))
reviewed_cluster_ids = set(approved_labels["cluster"])

assert not approved_labels["cluster"].duplicated().any()
assert reviewed_cluster_ids == current_cluster_ids

incomplete_approvals = approved_labels[
    (approved_labels["review_status"] != "approved")
    | (approved_labels["approved_cluster_subject_en"] == "")
]

if not incomplete_approvals.empty:
    raise RuntimeError(
        f"{len(incomplete_approvals)} English labels still require approval."
    )

approved_subject_en_map = approved_labels.set_index("cluster")[
    "approved_cluster_subject_en"
].to_dict()

print("Approved English labels:", len(approved_subject_en_map))


Approved English labels: 174


### 8. Translate approved labels into Norwegian

Translation uses the approved English wording together with Norwegian cluster
keywords and original Norwegian article titles. This avoids context-free literal
translations. Existing article translations and keyword translations are unchanged.


In [32]:
TRANSLATION_MODEL = "gpt-5.6-sol"
TRANSLATION_REASONING_EFFORT = "low"
TRANSLATION_PROMPT_VERSION = "approved-topic-label-no-v1"
TRANSLATION_CHECKPOINT_PATH = (
    DATA_DIR / "cluster_labels_no_gpt56sol_checkpoint.json"
)
TRANSLATED_LABELS_PATH = DATA_DIR / "cluster_labels_no_translated.csv"

TRANSLATION_INSTRUCTIONS = '''
You are a professional Norwegian newspaper taxonomy editor. Translate the
approved English cluster label into concise, idiomatic Norwegian Bokmål.
Use the Norwegian cluster keywords and representative original Norwegian
titles to resolve context. Preserve proper names and established Norwegian
institutional terminology. Translate the intended topic, not each English
word mechanically. Return only the Norwegian label in the structured field.
Do not broaden, narrow, explain, or relabel the approved English subject.
'''.strip()

norwegian_context_by_cluster = {}

for cluster_id, group in entities_manual.groupby("cluster", sort=True):
    representative_ids = {
        article["id"]
        for context in label_contexts
        if context["cluster"] == cluster_id
        for article in context["representative_articles"]
    }
    representative_rows = group[group["id"].isin(representative_ids)]

    norwegian_context_by_cluster[cluster_id] = {
        "cluster": cluster_id,
        "approved_cluster_subject_en": approved_subject_en_map[cluster_id],
        "top_keywords_no": parse_keyword_list(
            group.iloc[0]["top_keywords_no"]
        ),
        "representative_titles_no": representative_rows[
            "title_no"
        ].astype(str).tolist()
    }

if TRANSLATION_CHECKPOINT_PATH.exists():
    with TRANSLATION_CHECKPOINT_PATH.open(
        "r",
        encoding="utf-8"
    ) as input_file:
        translation_checkpoint = json.load(input_file)
else:
    translation_checkpoint = {}

translated_results = {}
failed_translation_clusters = []

for cluster_id in tqdm(
    sorted(norwegian_context_by_cluster),
    desc="Translating approved labels"
):
    context = norwegian_context_by_cluster[cluster_id]
    context_hash = stable_json_hash(context)
    existing = translation_checkpoint.get(cluster_id)

    if (
        isinstance(existing, dict)
        and existing.get("model") == TRANSLATION_MODEL
        and existing.get("prompt_version") == TRANSLATION_PROMPT_VERSION
        and existing.get("context_hash") == context_hash
        and existing.get("cluster_subject_no")
    ):
        translated_results[cluster_id] = existing
        continue

    result = None

    for attempt in range(1, 4):
        try:
            response = client.responses.create(
                model=TRANSLATION_MODEL,
                reasoning={"effort": TRANSLATION_REASONING_EFFORT},
                instructions=TRANSLATION_INSTRUCTIONS,
                input=json.dumps(context, ensure_ascii=False),
                text={
                    "format": {
                        "type": "json_schema",
                        "name": "norwegian_cluster_label",
                        "strict": True,
                        "schema": {
                            "type": "object",
                            "properties": {
                                "cluster_subject_no": {"type": "string"}
                            },
                            "required": ["cluster_subject_no"],
                            "additionalProperties": False
                        }
                    }
                },
                max_output_tokens=100
            )

            parsed = json.loads(response.output_text)
            norwegian_label = parsed["cluster_subject_no"].strip()

            if not norwegian_label:
                raise ValueError("The model returned an empty Norwegian label.")

            result = {
                "cluster": cluster_id,
                "cluster_subject_en": approved_subject_en_map[cluster_id],
                "cluster_subject_no": norwegian_label,
                "model": TRANSLATION_MODEL,
                "prompt_version": TRANSLATION_PROMPT_VERSION,
                "context_hash": context_hash
            }
            break

        except Exception as error:
            print(
                f"Attempt {attempt} failed for {cluster_id}: {error}"
            )
            if attempt < 3:
                time.sleep(2 ** attempt)

    if result is None:
        failed_translation_clusters.append(cluster_id)
        continue

    translation_checkpoint[cluster_id] = result
    translated_results[cluster_id] = result
    save_json_atomic(
        translation_checkpoint,
        TRANSLATION_CHECKPOINT_PATH
    )

if failed_translation_clusters:
    raise RuntimeError(
        "Norwegian translation failed for: "
        + ", ".join(sorted(failed_translation_clusters))
    )

translated_labels = pd.DataFrame(
    [translated_results[cluster_id] for cluster_id in sorted(translated_results)]
)

translated_labels.to_csv(
    TRANSLATED_LABELS_PATH,
    index=False,
    encoding="utf-8"
)

assert len(translated_labels) == 174
assert translated_labels["cluster_subject_no"].str.strip().ne("").all()

print("Translated Norwegian labels:", len(translated_labels))
print("Translation file:", TRANSLATED_LABELS_PATH)

translated_labels.head()


Translating approved labels:   0%|          | 0/174 [00:00<?, ?it/s]

Translated Norwegian labels: 174
Translation file: /Users/tina/Documents/GitHub/svalbardposten_weathermap_final/data/cluster_labels_no_translated.csv


,cluster,cluster_subject_en,cluster_subject_no,model,prompt_version,context_hash
0,n_0,Celestial phenomena,Himmelfenomener,gpt-5.6-sol,approved-topic-label-no-v1,8894c570de098bb22b0cad2414c2e85a529583b7a2b5b6...
1,n_1,Longyearbyen healthcare,Helsetjenester i Longyearbyen,gpt-5.6-sol,approved-topic-label-no-v1,88acb69f8efe396095690b10e5e5b634c12c1ddc9f5ba5...
2,n_10,Annual TV Telethon,TV-aksjonen,gpt-5.6-sol,approved-topic-label-no-v1,5adcf2854dc3264bd3f1f09fa01397d813b5901c64c5ef...
3,n_11,Local sports,Lokalidrett,gpt-5.6-sol,approved-topic-label-no-v1,ca14370c49d5f5ac51e33a75a3d1dfe2bcddf3d2c1231e...
4,n_12,Community Celebrations and Commemorations,Lokale feiringer og markeringer,gpt-5.6-sol,approved-topic-label-no-v1,8cbda71602ac5cf10a83da55adc8116a2b68ad8ea1c2e2...


In [33]:
translated_labels[
    translated_labels["cluster"].isin(
        label_revisions
    )
][
    [
        "cluster",
        "cluster_subject_en",
        "cluster_subject_no"
    ]
].sort_values("cluster")

,cluster,cluster_subject_en,cluster_subject_no
15,n_22,Retail and Hospitality Businesses,Handels- og serveringsbedrifter
23,n_3,Rabies and Animal Health,Rabies og dyrehelse
34,n_4,Space Research Infrastructure,Infrastruktur for romforskning
41,n_46,Maritime Operations and Incidents,Maritime operasjoner og hendelser
43,n_48,Rescue and Police Operations,Redningsaksjoner og politioperasjoner
54,p_1,Svalbard History Films,Filmer om Svalbards historie
73,p_116,Mark Sabbatini and Icepeople,Mark Sabbatini og Icepeople
78,p_120,Tourism Management and Sustainability,Reiselivsledelse og bærekraft
86,p_2,Old Hospital Property Disputes,Eiendomstvister om Gamle sykehuset
103,p_35,Revue and Cultural Awards,Revy og kulturpriser


In [34]:
norwegian_label_revisions = {
    "n_19": "Polarjazz",
    "n_21": "Utdanning og forskning ved UNIS",
    "n_22": "Handels-, overnattings- og serveringsbedrifter",
    "n_26": "Viltøkologi",
    "p_3": "Svalbard Globale frøhvelv",
    "p_23": "Driften av Svalbardbutikken",
    "p_30": "Lokalidrett",
    "p_34": "Livet i lokalsamfunnet",
    "p_39": "Polarjazz",
    "p_53": "Sysselmesteren på Svalbard",
    "p_56": "Stasjonene på Bjørnøya og Hopen",
    "p_58": "Marin forurensning",
    "p_63": "Isbjørn",
    "p_73": "Sjøsikkerhet og regelverk",
    "p_79": "Grunnstøtingen og bergingen av «Northguider»",
    "p_94": "Beredskap og krisehåndtering",
    "p_97": "Avfall og vraking av kjøretøy",
    "p_111": "Gruvenedleggelse og opprydding i Svea",
    "p_120": "Reiselivsforvaltning og bærekraft"
}

In [35]:
translated_labels.loc[
    translated_labels["cluster"].isin(norwegian_label_revisions),
    "cluster_subject_no"
] = (
    translated_labels.loc[
        translated_labels["cluster"].isin(norwegian_label_revisions),
        "cluster"
    ].map(norwegian_label_revisions)
)

assert len(translated_labels) == 174
assert translated_labels["cluster"].nunique() == 174
assert translated_labels["cluster_subject_no"].notna().all()
assert translated_labels["cluster_subject_no"].str.strip().ne("").all()

translated_labels.to_csv(
    TRANSLATED_LABELS_PATH,
    index=False,
    encoding="utf-8"
)

print("Norwegian labels verified:", len(translated_labels))
print("Norwegian labels revised:", len(norwegian_label_revisions))

Norwegian labels verified: 174
Norwegian labels revised: 19


In [37]:
translated_labels.loc[
    translated_labels["cluster"].isin(norwegian_label_revisions),
    "cluster_subject_no"
] = (
    translated_labels.loc[
        translated_labels["cluster"].isin(norwegian_label_revisions),
        "cluster"
    ].map(norwegian_label_revisions)
)

assert len(translated_labels) == 174
assert translated_labels["cluster"].nunique() == 174
assert translated_labels["cluster_subject_no"].notna().all()
assert translated_labels["cluster_subject_no"].str.strip().ne("").all()

translated_labels.to_csv(
    TRANSLATED_LABELS_PATH,
    index=False,
    encoding="utf-8"
)

print("Norwegian labels verified:", len(translated_labels))
print("Norwegian labels revised:", len(norwegian_label_revisions))

Norwegian labels verified: 174
Norwegian labels revised: 19


In [43]:
label_revisions_bilingual = {
    "n_50": {
        "en": "Police Incident Log",
        "no": "Politiloggen"
    },
    "p_54": {
        "en": "Police Incident Log",
        "no": "Politiloggen"
    },
    "n_52": {
        "en": "Hunting and Trapping Management",
        "no": "Forvaltning av jakt og fangst"
    },
    "p_49": {
        "en": "Police and Emergency Responses",
        "no": "Politi- og utrykningsoppdrag"
    },
    "p_119": {
        "en": "Sherpa-Built Trail Proposal",
        "no": "Forslag om sherpabygd sti"
    },
    "p_96": {
        "en": "Svalbard Policy White Paper",
        "no": "Svalbardmeldingen"
    },
    "p_66": {
        "en": "Governor of Svalbard Incident Log",
        "no": "Sysselmesterens hendelseslogg"
    },
    "p_117": {
        "en": "Governor of Svalbard Field Inspectors",
        "no": "Sysselmesterens feltinspektører"
    },
    "p_94": {
        "en": "Emergency Planning and Response",
        "no": "Beredskapsplanlegging og krisehåndtering"
    }
}

english_revisions = {
    cluster: labels["en"]
    for cluster, labels in label_revisions_bilingual.items()
}

norwegian_revisions = {
    cluster: labels["no"]
    for cluster, labels in label_revisions_bilingual.items()
}

# Update the approved English review file
approved_labels = pd.read_csv(
    LABEL_REVIEW_PATH,
    dtype={"cluster": str}
)

missing_review_clusters = (
    set(label_revisions_bilingual)
    - set(approved_labels["cluster"])
)
assert not missing_review_clusters, (
    f"Missing clusters in English review file: "
    f"{sorted(missing_review_clusters)}"
)

approved_labels.loc[
    approved_labels["cluster"].isin(english_revisions),
    "approved_cluster_subject_en"
] = (
    approved_labels.loc[
        approved_labels["cluster"].isin(english_revisions),
        "cluster"
    ].map(english_revisions)
)

approved_labels.loc[
    approved_labels["cluster"].isin(english_revisions),
    "review_notes"
] = "English label revised for international clarity."

approved_labels.to_csv(
    LABEL_REVIEW_PATH,
    index=False,
    encoding="utf-8"
)

# Update the English and Norwegian translation file
translated_labels = pd.read_csv(
    TRANSLATED_LABELS_PATH,
    dtype={"cluster": str}
)

missing_translation_clusters = (
    set(label_revisions_bilingual)
    - set(translated_labels["cluster"])
)
assert not missing_translation_clusters, (
    f"Missing clusters in translation file: "
    f"{sorted(missing_translation_clusters)}"
)

revision_mask = translated_labels["cluster"].isin(
    label_revisions_bilingual
)

translated_labels.loc[
    revision_mask,
    "cluster_subject_en"
] = (
    translated_labels.loc[revision_mask, "cluster"]
    .map(english_revisions)
)

translated_labels.loc[
    revision_mask,
    "cluster_subject_no"
] = (
    translated_labels.loc[revision_mask, "cluster"]
    .map(norwegian_revisions)
)

assert len(approved_labels) == 174
assert len(translated_labels) == 174
assert approved_labels["cluster"].nunique() == 174
assert translated_labels["cluster"].nunique() == 174
assert translated_labels["cluster_subject_en"].str.strip().ne("").all()
assert translated_labels["cluster_subject_no"].str.strip().ne("").all()

translated_labels.to_csv(
    TRANSLATED_LABELS_PATH,
    index=False,
    encoding="utf-8"
)

display(
    translated_labels.loc[
        revision_mask,
        [
            "cluster",
            "cluster_subject_en",
            "cluster_subject_no"
        ]
    ].sort_values("cluster")
)

print("English labels revised:", len(english_revisions))
print("Norwegian labels revised:", len(norwegian_revisions))

,cluster,cluster_subject_en,cluster_subject_no
46,n_50,Police Incident Log,Politiloggen
48,n_52,Hunting and Trapping Management,Forvaltning av jakt og fangst
74,p_117,Governor of Svalbard Field Inspectors,Sysselmesterens feltinspektører
76,p_119,Sherpa-Built Trail Proposal,Forslag om sherpabygd sti
118,p_49,Police and Emergency Responses,Politi- og utrykningsoppdrag
124,p_54,Police Incident Log,Politiloggen
137,p_66,Governor of Svalbard Incident Log,Sysselmesterens hendelseslogg
168,p_94,Emergency Planning and Response,Beredskapsplanlegging og krisehåndtering
170,p_96,Svalbard Policy White Paper,Svalbardmeldingen


English labels revised: 9
Norwegian labels revised: 9


### 9. Final validation and export

Set `CONFIRM_FINAL_EXPORT` to `True` only after reviewing both languages. The final
write updates `data/entities.csv` and `public/entities.csv`; the protected pre-review
backup is never touched.


In [44]:
CONFIRM_FINAL_EXPORT = True

if not CONFIRM_FINAL_EXPORT:
    raise RuntimeError(
        "Final export is paused. Review the labels, then set "
        "CONFIRM_FINAL_EXPORT = True and rerun this cell."
    )

# translated_subject_no_map = translated_labels.set_index("cluster")[
#     "cluster_subject_no"
# ].to_dict()
# Reload the final approved labels from disk so the export
# does not depend on stale in-memory dictionaries.
approved_labels = pd.read_csv(
    LABEL_REVIEW_PATH,
    dtype={"cluster": str}
)

translated_labels = pd.read_csv(
    TRANSLATED_LABELS_PATH,
    dtype={"cluster": str}
)

approved_subject_en_map = (
    approved_labels
    .set_index("cluster")["approved_cluster_subject_en"]
    .to_dict()
)

translated_subject_no_map = (
    translated_labels
    .set_index("cluster")["cluster_subject_no"]
    .to_dict()
)

assert len(approved_subject_en_map) == 174
assert len(translated_subject_no_map) == 174

entities_final = entities_manual.copy()
entities_final["cluster_subject_en"] = (
    entities_final["cluster"].map(approved_subject_en_map)
)
entities_final["cluster_subject_no"] = (
    entities_final["cluster"].map(translated_subject_no_map)
)

FINAL_ENTITY_COLUMNS = [
    "id",
    "created_by_name",
    "published",
    "published_url",
    "year",
    "month",
    "day",
    "title_no",
    "title_en",
    "subtitle_no",
    "subtitle_en",
    "excerpt_no",
    "excerpt_en",
    "tags_no",
    "tags_en",
    "word_count",
    "temperature",
    "color",
    "x",
    "y",
    "cluster",
    "article_keywords_no",
    "article_keywords_en",
    "top_keywords_no",
    "top_keywords_en",
    "cluster_subject_en",
    "cluster_subject_no"
]

missing_final_columns = set(FINAL_ENTITY_COLUMNS) - set(entities_final.columns)

if missing_final_columns:
    raise ValueError(
        f"Missing final columns: {sorted(missing_final_columns)}"
    )

entities_export = entities_final[FINAL_ENTITY_COLUMNS].copy()

assert len(entities_export) == EXPECTED_ROWS
assert entities_export.shape[1] == 27
assert entities_export["cluster"].nunique() == 174
assert not entities_export["id"].duplicated().any()
assert int(entities_export.isna().sum().sum()) == 0
assert entities_export["cluster_subject_en"].str.strip().ne("").all()
assert entities_export["cluster_subject_no"].str.strip().ne("").all()

final_split_counts = entities_export[
    entities_export["cluster"].isin(EXPECTED_SPLIT_COUNTS)
]["cluster"].value_counts().to_dict()
assert final_split_counts == EXPECTED_SPLIT_COUNTS

output_paths = [
    DATA_DIR / "entities.csv",
    PUBLIC_DIR / "entities.csv"
]

for output_path in output_paths:
    entities_export.to_csv(
        output_path,
        index=False,
        encoding="utf-8"
    )

exported_hashes = {
    str(output_path): file_sha256(output_path)
    for output_path in output_paths
}

assert len(set(exported_hashes.values())) == 1
assert BACKUP_ENTITIES_PATH.is_file()
assert file_sha256(BACKUP_ENTITIES_PATH) == backup_hash

print("Final exports complete")
print("Rows:", len(entities_export))
print("Columns:", len(entities_export.columns))
print("Clusters:", entities_export["cluster"].nunique())
print("Missing values:", int(entities_export.isna().sum().sum()))
print("Protected backup:", BACKUP_ENTITIES_PATH)
print("Files:")
for output_path, output_hash in exported_hashes.items():
    print("-", output_path, output_hash)


Final exports complete
Rows: 10946
Columns: 27
Clusters: 174
Missing values: 0
Protected backup: /Users/tina/Documents/GitHub/svalbardposten_weathermap_final/data/entities_before_manualcheck.csv
Files:
- /Users/tina/Documents/GitHub/svalbardposten_weathermap_final/data/entities.csv 211e00bc0c743e819f62b2e688734b928497fbab4d5fe1adefe3723820a9874b
- /Users/tina/Documents/GitHub/svalbardposten_weathermap_final/public/entities.csv 211e00bc0c743e819f62b2e688734b928497fbab4d5fe1adefe3723820a9874b


In [45]:
check = pd.read_csv(SOURCE_ENTITIES_PATH, dtype={"cluster": str})

expected = {
    "n_50": "Police Incident Log",
    "p_54": "Police Incident Log",
    "n_52": "Hunting and Trapping Management",
    "p_49": "Police and Emergency Responses",
    "p_119": "Sherpa-Built Trail Proposal",
    "p_96": "Svalbard Policy White Paper",
    "p_66": "Governor of Svalbard Incident Log",
    "p_117": "Governor of Svalbard Field Inspectors",
    "p_94": "Emergency Planning and Response"
}

actual = (
    check[check["cluster"].isin(expected)]
    .drop_duplicates("cluster")
    .set_index("cluster")["cluster_subject_en"]
)

assert actual.to_dict() == expected
print("All nine English revisions are present.")

All nine English revisions are present.
